# 🚀 vLLM Ungated & Uncensored GPU Server (Cloudflare Remote Tunnel)

This notebook turns your Google Colab instance into a high-performance **OpenAI-Compatible API Server** serving **ungated and uncensored models** (Dolphin, Hermes 3, Abliterated Llama).

### ✨ Key Advantages:
- **100% Ungated**: No Hugging Face tokens (`HF_TOKEN`) or gated approvals required. Downloads anonymously and instantly.
- **No Refusal Guardrails**: Unfiltered instruction-following for coding, cybersecurity, creative writing, and complex reasoning.
- **PagedAttention & Prefix Caching**: Maximal KV-cache reuse on NVIDIA T4 / A100 GPUs.
- **Cloudflare Quick Tunnel**: Exposes a secure, zero-setup public HTTPS endpoint for your local FastAPI Gateway and load tester.

## 1. Verify NVIDIA GPU Hardware
Ensure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU` or `A100 GPU`).

In [ ]:
!nvidia-smi

## 2. Install Dependencies & Colab Fixes
Installs latest vLLM, fixes PyTorch/Torchaudio version mismatches, and installs the Cloudflare Tunnel CLI.

In [ ]:
# 1. Fix Colab CUDA/TorchAudio mismatch
!pip uninstall -y torchaudio

# 2. Install latest vLLM
!pip install -q -U vllm

# 3. Download and install Cloudflare Tunnel CLI
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## 3. Select Model & Launch vLLM Server + Cloudflare Tunnel
Choose your desired uncensored model from the dropdown below and run the cell.

In [ ]:
#@title ⚙️ Model Selection & Server Configuration

MODEL_CHOICE = "cognitivecomputations/dolphin-2.9.3-qwen2-1.5b" #@param ["cognitivecomputations/dolphin-2.9.3-qwen2-1.5b", "failspy/Llama-3.2-3B-Instruct-abliterated", "NousResearch/Hermes-3-Llama-3.1-8B", "failspy/Meta-Llama-3.1-8B-Instruct-abliterated", "cognitivecomputations/dolphin-2.9.2-qwen2-7b"] {allow-input: true}
API_KEY = "dev-secret" #@param {type:"string"}
PORT = "8000" #@param {type:"string"}
MAX_MODEL_LEN = 4096 #@param {type:"integer"}
GPU_MEMORY_UTILIZATION = 0.90 #@param {type:"number"}

import os
import re
import subprocess
import sys
import time

print(f"🚀 Starting vLLM server with uncensored model: {MODEL_CHOICE}...")

# 1. Construct vLLM Command
vllm_cmd = [
    "vllm", "serve", MODEL_CHOICE,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--api-key", str(API_KEY),
    "--enable-prefix-caching",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
]

# Start vLLM process
vllm_proc = subprocess.Popen(vllm_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

vllm_ready = False
for line in vllm_proc.stdout:
    print(line, end="")
    if "Application startup complete" in line or "Uvicorn running on" in line:
        vllm_ready = True
        print("\n✅ vLLM Engine is UP and ready!\n")
        break

if not vllm_ready and vllm_proc.poll() is not None:
    print(f"\n❌ vLLM failed to start (exit code {vllm_proc.returncode}). Check traceback above.")
    sys.exit(1)

# 2. Start Cloudflare Quick Tunnel
print("🌐 Spawning Cloudflare Quick Tunnel...")
tunnel_cmd = ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for line in tunnel_proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "=" * 65)
        print(f"🎉 PUBLIC TUNNEL READY: {tunnel_url}")
        print("\n📝 Copy this into your local .env file:")
        print(f"VLLM_BASE_URL={tunnel_url}/v1")
        print(f"VLLM_API_KEY={API_KEY}")
        print(f"VLLM_MODEL={MODEL_CHOICE}")
        print("=" * 65 + "\n")
        break

# Keep cell alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down server and tunnel...")
    vllm_proc.terminate()
    tunnel_proc.terminate()
    print("Processes terminated.")

## 4. (Optional) In-Colab Zero-Network Latency Benchmark
Measure true GPU compute and continuous batching saturation directly inside Colab (bypassing internet transit latency).

In [ ]:
# Run vLLM's official benchmark against the active uncensored model
!python3 -m vllm.entrypoints.openai.bench_serving \
    --backend vllm \
    --model cognitivecomputations/dolphin-2.9.3-qwen2-1.5b \
    --endpoint /v1/chat/completions \
    --dataset-name random \
    --random-input-len 256 \
    --random-output-len 128 \
    --num-prompts 50 \
    --request-rate 10 \
    --host localhost \
    --port 8000 \
    --api-key dev-secret